- gmx File Utility


In [3]:
import os
import datetime
import shutil
import gzip
import csv
import sys
import fnmatch
import io
import json
from importlib_metadata import files
import pandas as pd
from azure.storage.blob import BlobClient, ContainerClient, __version__
import TELTools as TT

c:\Users\214225\AppData\Local\Programs\Python\Python37\lib\site-packages\snowflake\connector\vendored\requests\__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.6) or chardet (5.0.0)/charset_normalizer (2.0.9) doesn't match a supported version!
  RequestsDependencyWarning)


In [4]:
# utility functions
def getSASKeys(useReplayQueue,srcAccount):
    tgtAccount = "SFSA"

    if useReplayQueue == "true":
        folderList = ["QUEUE/"]
        sasKeySRC = "AZURE-STORAGE-GMX-SAS-" + "SFSA"
    else:
        folderList = ["US/","Common/"]
        sasKeySRC = "AZURE-STORAGE-GMX-SAS-" + srcAccount
        
    sasKeyTGT = "AZURE-STORAGE-GMX-SAS-" + tgtAccount
    #print(sasKeySRC,sasKeyTGT)

    sasURLSRC = TT.getAzureSecret(sasKeySRC).value
    sasURLTGT = TT.getAzureSecret(sasKeyTGT).value
    #print(sasURLSRC,sasURLTGT)

    if not bool(sasURLSRC) or not bool(sasURLTGT):
        print("One or both SAS key environment variables are empty")
    
    return sasURLSRC,sasURLTGT,folderList

def getFileConfig(file_uploader_json,newFileNameTXT):

    for fileConfig in file_uploader_json:
        fileConfigDict = file_uploader_json.get(fileConfig)
        fileMatchString = fileConfigDict.get('FILE')
        print(newFileNameTXT,fileMatchString)
        if fnmatch.fnmatch(newFileNameTXT,fileMatchString):
            return fileConfig

def writeFile(path,fileOut):
    f = open(path,'w')
    f.write(fileOut)
    f.close()



In [21]:
# load config entries from TELTools.json
tel_tools_path = 'C:\Source\Repos\EnterpriseCloud\Configuration\TELTools.json'
ENV = 'DEV'
def getTELToolsConfig():
    # load from teltools.json
    with open(tel_tools_path,'r') as f:
        tel_tools_json = json.load(f,strict=False)
        gmx_json = tel_tools_json.get('Services').get('GMX').get(ENV)
        FILEUPLOADER_CONFIG = gmx_json.get('FILEUPLOADER_CONFIG')
        THREADS = int(gmx_json.get('THREADS'))

    return FILEUPLOADER_CONFIG,THREADS


In [22]:
# read gmx config file

FILEUPLOADER_CONFIG, THREADS = getTELToolsConfig()
file_uploader_json =TT.getFileUploaderConfig(FILEUPLOADER_CONFIG)
print(file_uploader_json)

# for fileConfig in file_uploader_json:
#     print(file_uploader_json.get(fileConfig))


{'FILELIST': {'FILE': 'FILELIST_*', 'SNOWFLAKE_DATABASE_NAME': '<<LZ>>', 'SNOWFLAKE_ROLE_NAME': '<<SYSADMIN>>', 'SNOWFLAKE_SCHEMA_NAME': 'SAPH', 'SNOWFLAKE_TABLE_NAME': 'FILELIST', 'SNOWFLAKE_WAREHOUSE_NAME': 'Import_File_WH', 'SPLIT_FILE': False}, 'PARTS_EXPORTIMPORT_CONTROL': {'FILE': 'PARTS_EXPORTIMPORT_CONTROL_*', 'SNOWFLAKE_DATABASE_NAME': '<<LZ>>', 'SNOWFLAKE_ROLE_NAME': '<<SYSADMIN>>', 'SNOWFLAKE_SCHEMA_NAME': 'SAPH', 'SNOWFLAKE_TABLE_NAME': 'PARTS_EXPORTIMPORT_CONTROL', 'SNOWFLAKE_WAREHOUSE_NAME': 'Import_File_WH', 'SPLIT_FILE': False}, 'SF_SAP_AAM_EV_ANLB': {'FILE': 'SF_SAP_AAM_EV_ANLB_*', 'SNOWFLAKE_DATABASE_NAME': '<<LZ>>', 'SNOWFLAKE_ROLE_NAME': '<<SYSADMIN>>', 'SNOWFLAKE_SCHEMA_NAME': 'SAPH', 'SNOWFLAKE_TABLE_NAME': 'SAP_AAM_EV_ANLB', 'SNOWFLAKE_WAREHOUSE_NAME': 'Import_File_WH', 'SPLIT_FILE': False}, 'SF_SAP_AAM_EV_ANLBZW_GMX': {'FILE': 'SF_SAP_AAM_EV_ANLBZW_GMX_*', 'SNOWFLAKE_DATABASE_NAME': '<<LZ>>', 'SNOWFLAKE_ROLE_NAME': '<<SYSADMIN>>', 'SNOWFLAKE_SCHEMA_NAME': 'SAPH'

In [19]:
# load file list

with open('C:\\Users\\214225\\Documents\\Filelist_20230502060107.tsv') as f:
    df_files = pd.read_csv(f, sep = "\x01")

raw_list = df_files['ï»¿TableOrViewName'].to_list()

file_list = []
for val in raw_list:
    file_list.append(val.upper())

file_list.append('FILELIST')
file_list.append('PARTS_EXPORTIMPORT_CONTROL')

print(len(file_list))
print(file_list)

173
['SF_SAP_AAM_EV_ANLB', 'SF_SAP_AAM_EV_ANLBZW_GMX', 'SF_SAP_AAM_EV_ANLC_GMX', 'SF_SAP_AAM_EV_ANLH_GMX', 'SF_SAP_AAM_EV_ANLU_GMX', 'SF_SAP_AAM_EV_ANLZ_GMX', 'SF_SAP_AAM_EV_T090NAT_GMX', 'SF_SAP_AAM_EV_T093T_GMX', 'SF_SAP_AAT_EV_ANEK', 'SF_SAP_AAT_EV_ANEP', 'SF_SAP_COM_EV_AUFK', 'SF_SAP_COM_EV_CEPC', 'SF_SAP_COM_EV_CEPCT', 'SF_SAP_COM_EV_CSKS', 'SF_SAP_COM_EV_CSKT', 'SF_SAP_COM_EV_PRHI_GMX', 'SF_SAP_COM_EV_PROJ', 'SF_SAP_COM_EV_PRPS', 'SF_SAP_COM_EV_PRTE_GMX', 'SF_SAP_COM_EV_T006A_GMX', 'SF_SAP_COM_EV_TKA01', 'SF_SAP_COM_EV_TKA02', 'SF_SAP_FIM_EV_ANLA', 'SF_SAP_FIM_EV_BNKA', 'SF_SAP_FIM_EV_DFKKBPTAXNUM_GMX', 'SF_SAP_FIM_EV_KNB1', 'SF_SAP_FIM_EV_LFB1', 'SF_SAP_FIM_EV_LFBW_GMX', 'SF_SAP_FIM_EV_SKA1', 'SF_SAP_FIM_EV_SKAT', 'SF_SAP_FIM_EV_SKB1', 'SF_SAP_FIM_EV_T001', 'SF_SAP_FIM_EV_T007S_GMX', 'SF_SAP_FIM_EV_T052', 'SF_SAP_FIM_EV_T052S_GMX', 'SF_SAP_FIM_EV_T052U', 'SF_SAP_FIM_EV_T059ZT_GMX', 'SF_SAP_FIM_EV_T880', 'SF_SAP_FIM_EV_TABW', 'SF_SAP_FIM_EV_TABWT', 'SF_SAP_FIM_EV_TSKDT_GMX', 'SF_

In [20]:
# match file list to fileuploaderconfig
table_list = []
error_list = []
for f in file_list:
    # print(f)
    f = f.upper()
    # print(f)
    try:
        table_name = file_uploader_json.get(f).get('SNOWFLAKE_TABLE_NAME')
        table_list.append(table_name)
        # print(f,table_name)
    except Exception as e:
        print(f'file: {f} failed: {e}')
        error_list.append(f)


file: SF_SAP_COM_EV_PRTE_GMX failed: 'NoneType' object has no attribute 'get'
file: SF_SAP_MMM_EV_MARA_GMX failed: 'NoneType' object has no attribute 'get'
file: SF_SAP_SDM_EV_A901_GMX failed: 'NoneType' object has no attribute 'get'
file: SF_SAP_US_FIT_EV_ACDOCA failed: 'NoneType' object has no attribute 'get'
file: SF_SAP_US_SDT_EV_PRCD_ELEMENTS failed: 'NoneType' object has no attribute 'get'
file: SF_SAP_US_SDT_EV_VBPA_GMX failed: 'NoneType' object has no attribute 'get'
file: SF_SAP_ZZM_EV_BUT000_GMX failed: 'NoneType' object has no attribute 'get'
file: SF_SAP_ZZM_EV_INOB_GMX failed: 'NoneType' object has no attribute 'get'


In [13]:
print(error_list)
print(table_list)
print(len(table_list))
print(file_list)

[]
['SAP_AAM_EV_ANLB', 'SAP_AAM_EV_ANLBZW_GMX', 'SAP_AAM_EV_ANLC_GMX', 'SAP_AAM_EV_ANLH_GMX', 'SAP_AAM_EV_ANLU_GMX', 'SAP_AAM_EV_ANLZ_GMX', 'SAP_AAM_EV_T090NAT_GMX', 'SAP_AAM_EV_T093T_GMX', 'SAP_AAT_EV_ANEK', 'SAP_AAT_EV_ANEP', 'SAP_COM_EV_AUFK', 'SAP_COM_EV_CEPC', 'SAP_COM_EV_CEPCT', 'SAP_COM_EV_CSKS', 'SAP_COM_EV_CSKT', 'SAP_COM_EV_PRHI_GMX', 'SAP_COM_EV_PROJ', 'SAP_COM_EV_PRPS', 'SAP_COM_EV_PRTE', 'SAP_COM_EV_T006A_GMX', 'SAP_COM_EV_TKA01', 'SAP_COM_EV_TKA02', 'SAP_FIM_EV_ANLA', 'SAP_FIM_EV_BNKA', 'SAP_FIM_EV_DFKKBPTAXNUM_GMX', 'SAP_FIM_EV_KNB1', 'SAP_FIM_EV_LFB1', 'SAP_FIM_EV_LFBW_GMX', 'SAP_FIM_EV_SKA1', 'SAP_FIM_EV_SKAT', 'SAP_FIM_EV_SKB1', 'SAP_FIM_EV_T001', 'SAP_FIM_EV_T007S_GMX', 'SAP_FIM_EV_T052', 'SAP_FIM_EV_T052S_GMX', 'SAP_FIM_EV_T052U', 'SAP_FIM_EV_T059ZT_GMX', 'SAP_FIM_EV_T880', 'SAP_FIM_EV_TABW', 'SAP_FIM_EV_TABWT', 'SAP_FIM_EV_TSKDT_GMX', 'SAP_FIM_EV_TSKM_GMX', 'SAP_FIM_EV_TSKMT_GMX', 'SAP_FIM_EV_TSTL_GMX', 'SAP_FIT_EV_BSET', 'SAP_FIT_EV_ZFIT0034_GMX', 'SAP_MMM_EV_EINA

In [14]:
# reverse match fileuploaderconfig to file list
matches = []
exceptions = []
print(len(file_list))
print(len(file_uploader_json))
cnt = 0
for k in file_uploader_json:
    cnt += 1
    try:
        if k.upper() in file_list:
            matches.append(k)
          
        else:
            exceptions.append(k)
            # print(k)

    except Exception as e:
        print(f'file_uploader did not match: {k} failed: {e}')
        exceptions.append(k)

print(cnt)
print(matches)
print(exceptions)

173
173
173
['FILELIST', 'PARTS_EXPORTIMPORT_CONTROL', 'SF_SAP_AAM_EV_ANLB', 'SF_SAP_AAM_EV_ANLBZW_GMX', 'SF_SAP_AAM_EV_ANLC_GMX', 'SF_SAP_AAM_EV_ANLH_GMX', 'SF_SAP_AAM_EV_ANLU_GMX', 'SF_SAP_AAM_EV_ANLZ_GMX', 'SF_SAP_AAM_EV_T090NAT_GMX', 'SF_SAP_AAM_EV_T093T_GMX', 'SF_SAP_AAT_EV_ANEK', 'SF_SAP_AAT_EV_ANEP', 'SF_SAP_COM_EV_AUFK', 'SF_SAP_COM_EV_CEPC', 'SF_SAP_COM_EV_CEPCT', 'SF_SAP_COM_EV_CSKS', 'SF_SAP_COM_EV_CSKT', 'SF_SAP_COM_EV_PRHI_GMX', 'SF_SAP_COM_EV_PROJ', 'SF_SAP_COM_EV_PRPS', 'SF_SAP_COM_EV_PRTE_GMX', 'SF_SAP_COM_EV_T006A_GMX', 'SF_SAP_COM_EV_TKA01', 'SF_SAP_COM_EV_TKA02', 'SF_SAP_FIM_EV_ANLA', 'SF_SAP_FIM_EV_BNKA', 'SF_SAP_FIM_EV_DFKKBPTAXNUM_GMX', 'SF_SAP_FIM_EV_KNB1', 'SF_SAP_FIM_EV_LFB1', 'SF_SAP_FIM_EV_LFBW_GMX', 'SF_SAP_FIM_EV_SKA1', 'SF_SAP_FIM_EV_SKAT', 'SF_SAP_FIM_EV_SKB1', 'SF_SAP_FIM_EV_T001', 'SF_SAP_FIM_EV_T007S_GMX', 'SF_SAP_FIM_EV_T052', 'SF_SAP_FIM_EV_T052S_GMX', 'SF_SAP_FIM_EV_T052U', 'SF_SAP_FIM_EV_T059ZT_GMX', 'SF_SAP_FIM_EV_T880', 'SF_SAP_FIM_EV_TABW', 'SF_

In [ ]:
# Generate ddl files for LZ tables for target schema

targetSchema = 'SAPH'
df_tables = pd.DataFrame()
df_tables['TABLE_NAME'] = table_list

df_tables['ddl'] = """create or replace TABLE """ + targetSchema + "." + df_tables['TABLE_NAME'] + """ (\
_FILENAME VARCHAR(16777216),\
_FILEDATE TIMESTAMP_NTZ(9),\
_UPLOADDATE TIMESTAMP_NTZ(9),\
JSONFIELD VARIANT\
)COMMENT='Model GMX """ + df_tables['TABLE_NAME'] + """ file.';"""

#print(df_tables['ddl'].values)

## output the LZ tables to file
outstr = ""
for idx, row in df_tables.iterrows():
    outstr += row['ddl'] + """
"""
print(outstr)
path = "C:\\Source\\Repos\\EnterpriseCloud\\Snow\\K.22282_gmxLZTablesPROD"
writeFile(path,outstr)

In [9]:
# load ods to lz comparison file, tables to remove from ods
# load file list

with open('C:\\Users\\214225\\Downloads\\GMX_ODS_tables.csv',encoding='utf-8') as f:
    df_files = pd.read_csv(f)

delete_list = df_files['TABLE_NAME'].to_list()

print(len(delete_list))
print(delete_list)

73
['SAP_MMM_EV_PURCHASEGRP', 'SAP_FIM_EVF_TAX_GMX', 'SAP_ZZM_EVF_ADDRESS_GMX', 'SAP_MMM_EV_STORAGELOCATION', 'SAP_MMM_EV_MATERIALGRP', 'SAP_ZZM_EV_REGION', 'SAP_FIM_EV_ACCOUNT', 'SAP_COM_EVF_PROFITCENTERHIER_GMX', 'SAP_MMM_EVF_STORAGELOCATION_GMX', 'SAP_FIT_EVF_NEWGLTOTAL_GMX', 'SAP_INVENTORYFORMATERIAL_GMX', 'SAP_AAM_EVF_DEPRAREA_GMX', 'SAP_FIM_EVF_CUSTOMERTAX_GMX', 'SAP_ZZM_EV_CURRENCY', 'SAP_COM_EVF_COSTCENTERHIER_GMX', 'SAP_SDM_EVF_DOMAINFIXEDVAL_GMX', 'SAP_FIT_EVF_NEWGLDOCUMENT_GMX', 'SAP_COM_EVF_CONTROLAREAASSIGN_GMX', 'PARTS_LEDGER_ACTUALS', 'SAP_ZZM_EV_COUNTRY', 'SAP_MMM_EVF_VENDORATTRIBUTE_GMX', 'SAP_SDT_EVF_DELIVERYORDER_GMX', 'SAP_ZZM_EVF_ITNADDRESSVER_GMX', 'SAP_FIM_EV_COMPANY', 'SAP_SDM_EVF_CUSTOMERATTRIBUTE_GMX', 'SAP_COM_EV_ORDER', 'SAP_SDM_EV_BILLINGDOCTYPE', 'SALES_LEDGER_ACTUALS', 'SAP_MMM_EVF_VENDOR_GMX', 'SAP_SDT_EVF_SALESCONDITION_GMX', 'SAP_MMT_EVF_INVENTORY_GMX', 'SAP_MMM_EV_MATERIAL', 'SAP_SDT_EVF_SO_GTM_RECEIVED_GMX', 'SAP_SDM_EV_SALESGROUP', 'SAP_MMM_EVF_MODE

In [ ]:
# generate ddl files to remove ODS tables

targetDB = 'ODS_SIT'
targetSchema = 'SAPH'
df_tables = pd.DataFrame()
df_tables['TABLE_NAME'] = delete_list

df_tables['ddl'] = """DROP TABLE IF EXISTS """ + targetSchema + "." + df_tables['TABLE_NAME'] + """;"""

print(df_tables['ddl'].values)

# output the drop tables to file
outstr = ""
for idx, row in df_tables.iterrows():
    outstr += row['ddl'] + """
"""
print(outstr)
path = "C:\\Source\\Repos\\EnterpriseCloud\\Snow\\K.24282_gmxLZTablesDELETEODS"
writeFile(path,outstr)

In [ ]:
# remove unused entries from fileuplaoder_gmx

print(type(file_uploader_json))
print(len(file_uploader_json))
delete_models = []

for k in exceptions:
    try:
        table_name = file_uploader_json.get(k).get('SNOWFLAKE_TABLE_NAME')
        delete_models.append(table_name)
        del file_uploader_json[k]
        print(f'deleted {k}')

    except:
        print(f'exception not found in file_uploader: {k}')
        pass

print(len(file_uploader_json))

file_path = "C:\\Source\\Repos\\EnterpriseCloud\\Configuration\\FileUploader_GMX.json"
with open(file_path,'w',encoding='utf-8') as f:
    json.dump(file_uploader_json, f, sort_keys = True, indent = 4,ensure_ascii = False)

In [ ]:
# remote lz to ods dbt models using exceptions

dbt_path = 'C:\\Source\\Repos\\EnterpriseCloud\\dbt\\lz_to_ods\\models\\saph\\'

for k in exceptions:
    k = k.replace("SF_","")
    # print(k)
    delete_file = dbt_path + k + '.sql'
    print(delete_file)
    try:
        os.remove(delete_file)
        print(f'removed {delete_file}')
    except: 
        print(f'file not found {delete_file}')


In [25]:
# remote ods to pub dbt models using exceptions

dbt_path = 'C:\\Source\\Repos\\EnterpriseCloud\\dbt\\ods_to_pub\\models\\saph\\'

model_delete = ['SAP_MMM_EV_PURCHASEGRP',
'SAP_FIM_EVF_TAX_GMX',
'SAP_ZZM_EVF_ADDRESS_GMX',
'SAP_MMM_EV_STORAGELOCATION',
'SAP_MMM_EV_MATERIALGRP',
'SAP_ZZM_EV_REGION',
'SAP_FIM_EV_ACCOUNT',
'SAP_COM_EVF_PROFITCENTERHIER_GMX',
'SAP_MMM_EVF_STORAGELOCATION_GMX',
'SAP_FIT_EVF_NEWGLTOTAL_GMX',
'SAP_INVENTORYFORMATERIAL_GMX',
'SAP_AAM_EVF_DEPRAREA_GMX',
'SAP_FIM_EVF_CUSTOMERTAX_GMX',
'SAP_ZZM_EV_CURRENCY',
'SAP_COM_EVF_COSTCENTERHIER_GMX',
'SAP_SDM_EVF_DOMAINFIXEDVAL_GMX',
'SAP_FIT_EVF_NEWGLDOCUMENT_GMX',
'SAP_COM_EVF_CONTROLAREAASSIGN_GMX',
'PARTS_LEDGER_ACTUALS',
'SAP_ZZM_EV_COUNTRY',
'SAP_MMM_EVF_VENDORATTRIBUTE_GMX',
'SAP_SDT_EVF_DELIVERYORDER_GMX',
'SAP_ZZM_EVF_ITNADDRESSVER_GMX',
'SAP_FIM_EV_COMPANY',
'SAP_SDM_EVF_CUSTOMERATTRIBUTE_GMX',
'SAP_COM_EV_ORDER',
'SAP_SDM_EV_BILLINGDOCTYPE',
'SALES_LEDGER_ACTUALS',
'SAP_MMM_EVF_VENDOR_GMX',
'SAP_SDT_EVF_SALESCONDITION_GMX',
'SAP_MMT_EVF_INVENTORY_GMX',
'SAP_MMM_EV_MATERIAL',
'SAP_SDT_EVF_SO_GTM_RECEIVED_GMX',
'SAP_SDM_EV_SALESGROUP',
'SAP_MMM_EVF_MODELTYPE_GMX',
'SAP_MMM_EV_PURCHASEDOCTYPE',
'SAP_AAM_EVF_ASSETNUMBER_GMX',
'SAP_MMM_EV_VENDOR',
'SAP_FIM_EVF_TAXCLASSMATERIAL_GMX',
'SAP_SDT_EVF_DO_GTM_RECEIVED_GMX',
'SAP_MMM_EV_ACCCATEGORY',
'SAP_SDM_EV_ORDERREASON',
'SAP_FIM_EVF_PROCESSINGCATEGORY_GMX',
'SAP_US_MMT_EV_PRCDP_GMX',
'SAP_FIM_EVF_WITHHOLDINGTAX_GMX',
'SAP_MMM_EV_MOVINGTYPE',
'SAP_AAM_EVF_FIXEDASSET_GMX',
'SAP_FIM_EVF_PAYMENTTERMS_GMX',
'SAP_FIM_EV_CONSOLIDATECOMPANY',
'SAP_SDM_EV_SALESAREA',
'SAP_MMT_EVF_PURCHASEORDER_GMX',
'SAP_MMM_EV_MATERIALEX',
'SAP_FIT_EVF_NEWGLAPDOCUMENT_GMX',
'SAP_SDT_EVF_SALESORDER_GMX',
'SAP_SDM_EV_DOCSTATUS',
'SAP_SDM_EVF_CUSTOMER_GMX',
'SAP_MMM_EVF_PURCHASEREQUESTDOCTYPE_GMX',
'REPAIR_LEDGER_ACTUALS',
'SAP_COM_EVF_PROFITCENTERHIERYM_GMX',
'SAP_COM_EV_WBSELEMENT',
'SAP_COM_EVF_ASSIGNUOM_GMX',
'SAP_MMM_EV_PLANT',
'SAP_MMM_EVF_MATERIAL_GMX',
'SAP_ZZT_EVF_SCRIPTLONGTEXT_GMX',
'SAP_SDM_EV_SALESDOCITEMCAT',
'SO_ADJUSTMENT_LEDGER_ACTUALS',
'SAP_SDM_EV_SALESOFFICE',
'SAP_SDM_EV_SHIPPOINT',
'SAP_SDM_EVF_CONDITIONTYPE_GMX',
'SAP_FIM_EVF_PAYMENTTERMSDESCRIPTION_GMX',
'SAP_SDM_EV_SALESDOCTYPE',
'SAP_MMM_EV_ITEMCATEGORY',
'SAP_ZZM_EVF_CLASSSYSKWD_GMX']


print(model_delete)
for k in model_delete:
    k = k.replace("SF_","")
    k = 'v_' + k
    # print(k)
    delete_file = dbt_path + k + '.sql'
    print(delete_file)
    try:
        os.remove(delete_file)
        print(f'removed {delete_file}')
    except: 
        print(f'file not found {delete_file}')

['SAP_MMM_EV_PURCHASEGRP', 'SAP_FIM_EVF_TAX_GMX', 'SAP_ZZM_EVF_ADDRESS_GMX', 'SAP_MMM_EV_STORAGELOCATION', 'SAP_MMM_EV_MATERIALGRP', 'SAP_ZZM_EV_REGION', 'SAP_FIM_EV_ACCOUNT', 'SAP_COM_EVF_PROFITCENTERHIER_GMX', 'SAP_MMM_EVF_STORAGELOCATION_GMX', 'SAP_FIT_EVF_NEWGLTOTAL_GMX', 'SAP_INVENTORYFORMATERIAL_GMX', 'SAP_AAM_EVF_DEPRAREA_GMX', 'SAP_FIM_EVF_CUSTOMERTAX_GMX', 'SAP_ZZM_EV_CURRENCY', 'SAP_COM_EVF_COSTCENTERHIER_GMX', 'SAP_SDM_EVF_DOMAINFIXEDVAL_GMX', 'SAP_FIT_EVF_NEWGLDOCUMENT_GMX', 'SAP_COM_EVF_CONTROLAREAASSIGN_GMX', 'PARTS_LEDGER_ACTUALS', 'SAP_ZZM_EV_COUNTRY', 'SAP_MMM_EVF_VENDORATTRIBUTE_GMX', 'SAP_SDT_EVF_DELIVERYORDER_GMX', 'SAP_ZZM_EVF_ITNADDRESSVER_GMX', 'SAP_FIM_EV_COMPANY', 'SAP_SDM_EVF_CUSTOMERATTRIBUTE_GMX', 'SAP_COM_EV_ORDER', 'SAP_SDM_EV_BILLINGDOCTYPE', 'SALES_LEDGER_ACTUALS', 'SAP_MMM_EVF_VENDOR_GMX', 'SAP_SDT_EVF_SALESCONDITION_GMX', 'SAP_MMT_EVF_INVENTORY_GMX', 'SAP_MMM_EV_MATERIAL', 'SAP_SDT_EVF_SO_GTM_RECEIVED_GMX', 'SAP_SDM_EV_SALESGROUP', 'SAP_MMM_EVF_MODELTY

In [10]:
# load working variables
srcAccount = 'USR2'
useReplayQueue = 'no'

## vars
filesUnprocessed = "None"
filesProcessed = "FileName,RowCount\n"
serverPath = "F:\\Shares\\Files\\GMX\\"
targetPath = serverPath + srcAccount
errorPath = "ERROR/"
archivePath = "ARCHIVE/"
print("Target path: " + srcAccount + ", Server path: " + serverPath)

# get SAS keys from env variables
sasURLSRC,sasURLTGT,folderList = getSASKeys(useReplayQueue,srcAccount)

# create container clients
srcContainer = ContainerClient.from_container_url(sasURLSRC)
tgtContainer = ContainerClient.from_container_url(sasURLTGT)


Target path: USR2, Server path: F:\Shares\Files\GMX\


In [11]:
for folder in folderList:
    blobs_list = ContainerClient.list_blobs(srcContainer,name_starts_with= folder)
    #print(blobs_list)
    for blob in blobs_list:

        # build lists for threading
        # print("Started processing file: " + blob.name + '\n')
        rowCount = 0
        fileName = blob.name.replace("QUEUE/","")
        newFileNameGZ = fileName.replace("Common/","")
        newFileNameGZ = newFileNameGZ.replace("US/","")
        newFileNameTXT = newFileNameGZ.replace(".gz","").upper()
        downloadFilePath = os.path.join(targetPath, newFileNameGZ)
        targetFilePath = os.path.join(targetPath, newFileNameTXT)
        archiveFilePath = archivePath + fileName
        errorFilePath = errorPath + fileName

        break

In [12]:
# load file uploader
print(newFileNameTXT)

filekey = getFileConfig(file_uploader_json,newFileNameTXT)
print(filekey)

PARTS_EXPORTIMPORT_CONTROL_20230420130108.TXT
PARTS_EXPORTIMPORT_CONTROL_20230420130108.TXT FILELIST_*
PARTS_EXPORTIMPORT_CONTROL_20230420130108.TXT PARTS_EXPORTIMPORT_CONTROL_*
PARTS_EXPORTIMPORT_CONTROL


In [ ]:
# move this to a processBlob function
# download .gz blob and write to local
# blobContent = getBlob(blob,srcContainer,downloadFilePath)
blobContent = ContainerClient.download_blob(srcContainer,blob.name).readall()
blobBytesIO = io.BytesIO(blobContent)
with gzip.open(blobBytesIO) as gz:
    f = gz.read()
    blobDecompress = f.decode('utf8')
    blobString = io.StringIO(blobDecompress)

chunkCount = 0
totalRowCount = 0
for fileChunk in pd.read_csv(blobString, sep = "\x01", chunksize=1000000, engine="python", encoding='UTF_8_sig', quoting=csv.QUOTE_NONE):
    chunkCount += 1
    rowCount = len(fileChunk.index)
    totalRowCount += rowCount
    json_list= fileChunk.apply(lambda x: json.loads(x.to_json()), axis=1).to_list()
    print(json_list)
    
    # need to read config file to get the filenames and fileuploader config to get table names
    # snowflake_write_failures += TT.post_to_snowflake(cds_view,json_list,file_name,file_uploader_json,row_count)  

    break

In [ ]:
# df = pd.read_csv(blobDecompress, sep = "\x01", chunksize=1000000, engine="python", encoding='UTF_8_sig', quoting=csv.QUOTE_NONE)
print(json_list)



In [ ]:

#fileData = pd.DataFrame()
chunkCount = 0
rowCount = 0
for fileChunk in pd.read_csv(blobString, sep = "\x01", chunksize=1000000, engine="python", encoding='UTF_8_sig', quoting=csv.QUOTE_NONE):
    chunkCount += 1
    rowCount += len(fileChunk.index)
    # jsonFilePathOut = jsonFilePath + "-" + str(chunkCount) + ".json"
    df = fileChunk.to_json(orient = "records")
    print(df.head(10))
    break




In [ ]:
# TODO move files fitting a date pattern to the queue
%run ..\\Python\\getGMXFile.py "USR2" True

In [ ]:
# TODO refactor getGMXfile.py to pull from the queue if passed an optional parametr